The goal of this project is to experiment with YOLO and learn what we can do with it. We can also try to reinforce the application with the use of TensorRT given that this model will be running on a Jetson AGX Orin, or theoretically any parallel compute

# Imports

In [1]:
import time
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics import YOLO

print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")

torch 2.2.2, cuda available: False


# Dataset - FSOCO Sample

In [2]:
%pip install -q remotezip


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import sys
from pathlib import Path

from remotezip import RemoteZip

sys.path.append(str(Path("../../ml").resolve()))
from prepare_data import FSOCO_BBOX_URL, CLASSES

SAMPLE_SIZE = 200        # number of images to pull — tune as needed
VAL_FRACTION = 0.1
RANDOM_SEED = 42
OUT_DIR = Path("../../ml/data/fsoco_sample")

with RemoteZip(FSOCO_BBOX_URL) as zf:
    names = zf.namelist()

print(f"{len(names)} entries in the remote archive")
print(*names[:10], sep="\n")

Check the printed paths above against what the code assumes.

In [ ]:
meta_candidates = [n for n in names if n.endswith("meta.json")]
print(meta_candidates)

with RemoteZip(FSOCO_BBOX_URL) as zf:
    meta = json.loads(zf.read(meta_candidates[0]))

print("Using classes:", CLASSES)
print("(available in meta.json:", [c["title"] for c in meta["classes"]], ")")

Let's take a look at the images and how they're organized in FSOCO

In [ ]:
from prepare_data import prepare

# Same function used for the large-scale run on the Jetson — the notebook
# and the production data-prep share one implementation instead of drifting.
prepare(out_dir=OUT_DIR, sample_size=SAMPLE_SIZE, val_fraction=VAL_FRACTION, seed=RANDOM_SEED)

# Baseline COCO Weights

In [ ]:
model = YOLO("yolo26n.pt") # downloads pretrained weights
test_img = next((OUT_DIR / "train" / "images").glob("*.jpg"))
print(test_img)
results = model(str(test_img))
results[0].show()

# Train fine-tuned cone model

Runs the actual training via `ml/train.py`, against the FSOCO sample from
above (`prepare_data.prepare()` already wrote a real train/val split into
`dataset.yaml`). `epochs`/`imgsz` here are a fast CPU smoke test, not a real
training config — bump them up (and get a bigger sample / real GPU) once
you're doing an actual training run.

In [ ]:
from train import train

train(
    data_yaml=str(OUT_DIR / "dataset.yaml"),
    model_variant="yolo26n.pt",
    profile="auto",  # picks "smoke" (CPU) or "full" (CUDA, e.g. the Jetson) automatically
    project=str(Path("../../ml/runs/detect").resolve()),
    name="train",
)

# Load fine-tuned cone model

In [11]:
cone_model = YOLO("../../ml/runs/detect/train/weights/best.pt")
cone_model.names # expect {0: blue, 1: yellow, 2: orange, 3: large_orange}

results = cone_model(str(test_img))
results[0].show()


image 1/1 /Users/songyueli/Documents/GitHub/shabang/formula_driverless/perception/notebooks/../../ml/data/fsoco_sample/images/prom_00076.jpg: 416x640 (no detections), 159.2ms
Speed: 3.1ms preprocess, 159.2ms inference, 0.3ms postprocess per image at shape (1, 3, 416, 640)


# Inference latency benchmarking

In [12]:
def benchmark(model, img, n=50, warmup=5, **predict_kwargs):
    for _ in range(warmup):
        model(img, verbose=False, **predict_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(n):
        model(img, verbose=False, **predict_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return elapsed / n * 1000

ms = benchmark(cone_model, str(test_img))
print(f"{ms:.2f} ms/frame  ({1000/ms:.1f} FPS)")

137.52 ms/frame  (7.3 FPS)


# Model size sweep - Accuracy vs latency

In [13]:
variants = ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt"] 
for v in variants:
    m = YOLO(v)
    ms = benchmark(m, str(test_img))
    print(f"{v:15s} {ms:6.2f} ms/frame  ({1000/ms:5.1f} FPS)")

yolo26n.pt      141.85 ms/frame  (  7.0 FPS)
yolo26s.pt      258.21 ms/frame  (  3.9 FPS)
yolo26m.pt      542.54 ms/frame  (  1.8 FPS)


# Precision and Export Optimizations

In [ ]:
# FP16
ms_fp16 = benchmark(cone_model, str(test_img), quantize=16) # GPU-only — expect no real speedup on this CPU-only Mac
print(f"FP16: {ms_fp16:.2f} ms/frame  ({1000/ms_fp16:.1f} FPS)")

# ONNX export
cone_model.export(format="onnx", imgsz=1440, simplify=True)

import onnxruntime as ort
sess = ort.InferenceSession("../../ml/runs/detect/train/weights/best.onnx",
                            providers = ["CPUExecutionProvider"]) # or CUDAExecutionProvider

# build an onnxruntime benchmark loop analogous to 'benchmark()' above

# Results & Analysis